# Gap Fade Strategy Demonstration

This notebook mirrors `qc_projects/main_gap_fade.py`, a QuantConnect Lean algorithm that fades large overnight gaps in SPY. We:
- Detect ±2% opening gaps relative to the previous close
- Enter mean-reversion trades (short gap-ups, long gap-downs)
- Approximate the intraday stop (breaching the gap extreme) using daily highs/lows
- Exit after the intended 120-minute hold if the stop does not trigger

Because Yahoo Finance only provides daily aggregates for long lookbacks, we treat the session high/low as proxies for intraday stop behavior and exit at the daily close when the time stop fires.

## Step 1: Imports and Parameters

We preserve the QuantConnect defaults: SPY universe, 2020-01-01 → 2022-01-01 window, ±2% gap trigger, 120-minute hold, and a 2-day warmup to establish the previous close.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", None)

symbol = "SPY"
start_date = "2020-01-01"
end_date = "2022-01-01"
gap_threshold = 0.02  # 2%
hold_minutes = 120
trading_minutes = 390  # regular session for scaling
warmup_days = 2

print(f"Symbol: {symbol} | Range: {start_date} → {end_date}")
print(f"Gap trigger: {gap_threshold:.2%} | Hold: {hold_minutes} minutes")

Symbol: SPY | Range: 2020-01-01 → 2022-01-01
Gap trigger: 2.00% | Hold: 120 minutes


## Step 2: Download Daily OHLCV

We mimic the other notebooks' Yahoo Finance fetch (with `multi_level_index=False`) to stay consistent with the rest of the workspace.

In [2]:
price_df = yf.download(
    symbol,
    start=start_date,
    end=end_date,
    interval="1d",
    progress=False,
    multi_level_index=False,
)
price_df = price_df.dropna().tz_localize(None)
price_df.index.name = "date"

print(f"Fetched {len(price_df):,} daily bars")
price_df.head()

C:\Users\User\AppData\Local\Temp\ipykernel_52644\471923477.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_df = yf.download(


Fetched 505 daily bars


,Close,High,Low,Open,Volume
date,,,,,
2020-01-02,297.699005,297.717350,295.554718,296.480254,59151200
2020-01-03,295.444702,296.571839,294.244268,294.299248,77709700
2020-01-06,296.571899,296.654369,293.566200,293.685332,55653900
2020-01-07,295.738007,296.480259,295.288969,296.003732,40496400
2020-01-08,297.314117,298.532868,295.682992,295.930400,68296000


## Step 3: Compute Gap Features

We align with the QC logic by tracking the prior close, calculating the percentage gap at the open, and remembering whether it's an up-gap (fade via short) or down-gap (fade via long).

In [3]:
gap_df = price_df.copy()
gap_df["prev_close"] = gap_df["Close"].shift(1)
gap_df = gap_df.dropna(subset=["prev_close"]).copy()
gap_df["gap_pct"] = (gap_df["Open"] - gap_df["prev_close"]) / gap_df["prev_close"]
gap_df["gap_direction"] = np.where(gap_df["gap_pct"] > 0, "up", "down")
gap_df.loc[gap_df["gap_pct"].abs() < gap_threshold, "gap_direction"] = "none"
gap_df["entry_side"] = np.where(
    gap_df["gap_direction"] == "up",
    "short",
    np.where(gap_df["gap_direction"] == "down", "long", "flat"),
)

print(
    gap_df[["Open", "prev_close", "gap_pct", "gap_direction", "entry_side"]]
    .head(10)
)
print(
    gap_df["gap_direction"].value_counts()
)

                  Open  prev_close   gap_pct gap_direction entry_side
date                                                                 
2020-01-03  294.299248  297.699005 -0.011420          none       flat
2020-01-06  293.685332  295.444702 -0.005955          none       flat
2020-01-07  296.003732  296.571899 -0.001916          none       flat
2020-01-08  295.930400  295.738007  0.000651          none       flat
2020-01-09  298.881100  297.314117  0.005270          none       flat
2020-01-10  299.916589  299.330109  0.001959          none       flat
2020-01-13  299.091899  298.468719  0.002088          none       flat
2020-01-14  300.081519  300.521423 -0.001464          none       flat
2020-01-15  299.971561  300.063202 -0.000305          none       flat
2020-01-16  302.125047  300.741302  0.004601          none       flat
gap_direction
none    477
down     14
up       13
Name: count, dtype: int64


## Step 4: Simulate the Gap Fade Logic

We replicate the Lean loop with daily data approximations:
- **Entry:** at the official open when the gap exceeds ±2%
- **Stop:** hit if the day's extreme moves beyond the gap extreme (High for shorts, Low for longs)
- **Time Exit:** otherwise we exit at the close after ~120 minutes (scaled as 120/390 of the daily range)

In [4]:
time_fraction = min(hold_minutes / trading_minutes, 1.0)
trade_records = []
signals = []

for trade_date, row in gap_df.iterrows():
    if row["entry_side"] == "flat":
        continue

    entry_price = row["Open"]
    direction = row["entry_side"]
    stop_hit = False
    exit_reason = "time"
    exit_price = None
    minutes_held = hold_minutes

    if direction == "short":
        stop_hit = row["High"] > entry_price
        if stop_hit:
            exit_price = row["High"]
            exit_reason = "stop"
            minutes_held = np.nan
        else:
            exit_price = entry_price + (row["Close"] - row["Open"]) * time_fraction
    else:  # long
        stop_hit = row["Low"] < entry_price
        if stop_hit:
            exit_price = row["Low"]
            exit_reason = "stop"
            minutes_held = np.nan
        else:
            exit_price = entry_price + (row["Close"] - row["Open"]) * time_fraction

    if exit_price is None or not np.isfinite(exit_price):
        continue

    if direction == "short":
        ret_pct = (entry_price - exit_price) / entry_price * 100
    else:
        ret_pct = (exit_price - entry_price) / entry_price * 100

    trade_records.append({
        "date": trade_date.date().isoformat(),
        "direction": direction,
        "gap_pct": row["gap_pct"],
        "entry_price": round(entry_price, 2),
        "exit_price": round(exit_price, 2),
        "return_pct": round(ret_pct, 2),
        "stop_hit": stop_hit,
        "exit_reason": exit_reason,
        "minutes_held": minutes_held,
    })
    signals.append({
        "date": trade_date,
        "gap_pct": row["gap_pct"],
        "direction": direction,
        "return_pct": ret_pct,
        "stop_hit": stop_hit,
    })

trades_df = pd.DataFrame(trade_records)
signals_df = pd.DataFrame(signals)
print(f"Trades simulated: {len(trades_df)}")
trades_df.head() if not trades_df.empty else "No trades met the threshold"

Trades simulated: 27


,date,direction,gap_pct,entry_price,exit_price,return_pct,stop_hit,exit_reason,minutes_held
0,2020-02-24,long,-0.031006,296.11,294.37,-0.59,True,stop,NaN
1,2020-02-28,long,-0.029613,264.55,261.66,-1.09,True,stop,NaN
2,2020-03-05,long,-0.025187,279.47,274.92,-1.63,True,stop,NaN
3,2020-03-06,long,-0.030781,268.63,265.96,-1.00,True,stop,NaN
4,2020-03-09,long,-0.074498,252.27,250.58,-0.67,True,stop,NaN


## Step 5: Aggregate Performance

We track trade counts, win rate, average return, and stop frequency to gauge how the approximation behaves before porting tweaks back to Lean.

In [5]:
if trades_df.empty:
    summary = {
        "total_trades": 0,
        "win_rate_pct": 0.0,
        "avg_return_pct": 0.0,
        "median_return_pct": 0.0,
        "stop_rate_pct": 0.0,
    }
    per_direction = pd.DataFrame()
else:
    summary = {
        "total_trades": len(trades_df),
        "win_rate_pct": round((trades_df["return_pct"] > 0).mean() * 100, 1),
        "avg_return_pct": round(trades_df["return_pct"].mean(), 2),
        "median_return_pct": round(trades_df["return_pct"].median(), 2),
        "stop_rate_pct": round(trades_df["stop_hit"].mean() * 100, 1),
    }
    per_direction = trades_df.groupby("direction")["return_pct"].agg(
        trades="count",
        win_rate_pct=lambda s: round((s > 0).mean() * 100, 1),
        avg_return_pct=lambda s: round(s.mean(), 2),
        median_return_pct=lambda s: round(s.median(), 2),
    )

summary, per_direction

({'total_trades': 27,
  'win_rate_pct': np.float64(0.0),
  'avg_return_pct': np.float64(-1.66),
  'median_return_pct': np.float64(-1.15),
  'stop_rate_pct': np.float64(100.0)},
            trades  win_rate_pct  avg_return_pct  median_return_pct
 direction                                                         
 long           14           0.0           -1.73              -1.34
 short          13           0.0           -1.59              -1.15)

## Step 6: Visualize Gaps and Returns

Scatter gaps over time (color = P&L) and display the return distribution. Save under `graphs/gap_fade_signals.html` like the other demos.

In [6]:
graphs_dir = Path("graphs")
graphs_dir.mkdir(exist_ok=True)

if signals_df.empty:
    print("No gap trades were generated; skipping plot export.")
else:
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "Gap Size vs Date",
        "Return Distribution"
    ))

    fig.add_trace(
        go.Scatter(
            x=signals_df["date"],
            y=signals_df["gap_pct"],
            mode="markers",
            marker=dict(
                size=10,
                color=signals_df["return_pct"],
                colorscale="RdYlGn",
                showscale=True,
            ),
            text=[f"{d}: {ret:.2f}%" for d, ret in zip(signals_df["direction"], signals_df["return_pct"])],
            name="Gap events",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Histogram(x=signals_df["return_pct"], nbinsx=20, name="Returns"),
        row=1,
        col=2,
    )

    fig.update_layout(height=450, title="Gap Fade Performance Snapshot")
    output_path = graphs_dir / "gap_fade_signals.html"
    fig.write_html(output_path)
    fig.show()
    print(f"Saved interactive figure to {output_path}")

Saved interactive figure to graphs\gap_fade_signals.html


## Step 7: Inspect Trade Log

Surface the final trades to understand which sessions drove performance or triggered stops.

In [7]:
if trades_df.empty:
    print("No trades to display under current parameters.")
else:
    display(trades_df.tail(10))

,date,direction,gap_pct,entry_price,exit_price,return_pct,stop_hit,exit_reason,minutes_held
17,2020-04-15,long,-0.021917,255.85,253.91,-0.76,True,stop,NaN
18,2020-04-17,short,0.022501,263.05,264.82,-0.67,True,stop,NaN
19,2020-04-29,short,0.020299,268.72,271.81,-1.15,True,stop,NaN
20,2020-05-18,short,0.023648,270.12,273.53,-1.26,True,stop,NaN
21,2020-05-26,short,0.021968,278.30,278.54,-0.09,True,stop,NaN
22,2020-06-11,long,-0.023637,287.09,276.53,-3.68,True,stop,NaN
23,2020-06-12,short,0.025382,284.12,284.89,-0.27,True,stop,NaN
24,2020-06-15,long,-0.020348,274.70,273.52,-0.43,True,stop,NaN
25,2020-06-16,short,0.027455,290.79,290.94,-0.05,True,stop,NaN
26,2020-11-09,short,0.039439,338.32,338.70,-0.11,True,stop,NaN


## Recap

- Logic mirrors `qc_projects/main_gap_fade.py`, honoring the same gap triggers, fade directions, and 120-minute hold.
- Daily data approximations still reveal whether the idea adds value before running more expensive Lean backtests.
- Tweak `gap_threshold`, `hold_minutes`, or swap `symbol` to vet alternative parameter sets, then port the changes into the QuantConnect project.